# [데모 B] 코드 구조 리뷰형 — Conditional Edge

> **교안 형식 후보 B.** 완성된 원본 코드를 먼저 **구조로 읽고**, 슬라이드는 근거로 옆에 띄운다. 실행은 "어느 노드를 어떤 순서로 지났나"를 확인하는 데 쓴다. 11차수에서 호평받은 구조 분석·리뷰 중심 설명을 형식으로 만든 것이다.

| | |
|---|---|
| 커리큘럼 | 2.2 LangGraph 정통 이론 & 고급 기능 실습 — Conditional Edge로 분기 제어 |
| 슬라이드 | `docs/raw/Chapter01/day02/2-1-langgraph-theory-and-advanced-features.pdf` p5, p7~10 |
| 실습 원본 | `src/raw/Chapter01/04-langgraph/02_conditional_edge.py` |
| 실행 조건 | LLM을 호출한다. 입력 하나당 두 번(언어 감지 + 답변). |

## 왜 배우나 · 언제 쓰나

- LangChain 체인은 입력에서 출력까지 **선형** 흐름에 강하다. LangGraph는 **조건부 라우팅으로 다음 노드를 고르는** 흐름에 강하다 (슬라이드 p5).
- 이 예제의 분기는 "한국어냐 영어냐"지만, 같은 장치가 Agent에서는 "도구를 더 쓸지, 답을 낼지"를 정한다 (p8).

In [ ]:
from assets.widgets import show_checklist, show_code_map, show_graph_run, show_slides

## 1. 코드 지도 — 원본 전체를 구획으로 읽기

원본 파일을 그대로 읽어 구획별로 색을 입혔다. 구획을 누르면 설명과 대응 슬라이드가 나온다.
**강사 진행:** 위에서 아래로 구획을 넘기며 "이 부분은 무엇을 정의하나"를 먼저 묻고 답한다.

In [ ]:
show_code_map(
    "../src/raw/Chapter01/04-langgraph/02_conditional_edge.py",
    deck="2-1",
    sections=[
        {
            "title": "준비",
            "marker": "from typing import",
            "note": "LLM 인스턴스 하나를 모든 노드가 같이 쓴다.",
        },
        {
            "title": "State",
            "marker": "class State",
            "note": "그래프 전체가 공유하는 데이터다.\nmessages는 add_messages Reducer로 누적되고, language는 Reducer가 없어 새 값으로 덮어쓴다.",
            "page": 10,
        },
        {
            "title": "언어 감지 노드",
            "marker": "def detect_language",
            "note": "노드는 State를 받아 바꿀 부분만 dict로 돌려주는 함수다. 여기서는 language만 돌려준다.\n\n리뷰 질문: 'ko' in result.content.lower() 판별은 모델이 형식을 어기면 어떻게 되나?",
            "page": 7,
        },
        {
            "title": "응답 노드 2개",
            "marker": "def korean_response",
            "note": "system 메시지만 다르고 구조는 같다. messages로 돌려주므로 Reducer가 기존 대화 뒤에 붙인다.",
        },
        {
            "title": "라우팅 함수",
            "marker": "# ── 라우팅 함수",
            "note": "State를 보고 다음 노드의 '이름'을 돌려준다.\n반환 타입의 Literal이 갈 수 있는 노드 목록을 그래프에 알려 준다.",
            "page": 8,
        },
        {
            "title": "그래프 조립",
            "marker": "# ── 그래프 조립",
            "note": "add_node → START 연결 → add_conditional_edges → END.\n\n리뷰 질문: 주석 처리된 '병렬처리' 두 줄로 바꾸면 무엇이 달라지나? 조건부 엣지는 둘 중 하나로, 일반 엣지 두 개는 둘 다로 간다.",
            "page": 7,
        },
        {
            "title": "실행",
            "marker": "# ── 실행",
            "note": "invoke에 초기 State를 넣는다. 한국어·영어 질문을 하나씩 넣어 결과와 감지 언어를 출력한다.",
        },
    ],
)

## 2. 원본 코드 실행

원본에서 실행부를 뺀 나머지다. 원본을 고치면 이 셀도 함께 맞춘다.

In [ ]:
from typing import Annotated, Literal

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from typing_extensions import TypedDict

from config.llm import get_llm

load_dotenv()

llm = get_llm()


class State(TypedDict):
    messages: Annotated[list, add_messages]
    language: str  # 감지된 언어 ('ko' | 'en')


def detect_language(state: State) -> dict:
    """마지막 메시지의 언어를 감지해 State에 기록한다."""
    last = state["messages"][-1].content
    result = llm.invoke(
        f'다음 텍스트의 언어를 "ko" 또는 "en" 중 하나로만 판별해: {last}'
    )
    lang = "ko" if "ko" in result.content.lower() else "en"
    return {"language": lang}


def korean_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "반드시 한국어로만 답변하세요."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


def english_response(state: State) -> dict:
    response = llm.invoke(
        [
            ("system", "Respond only in English."),
            *state["messages"],
        ]
    )
    return {"messages": [response]}


def route_by_language(state: State) -> Literal["korean_response", "english_response"]:
    return "korean_response" if state.get("language") == "ko" else "english_response"


builder = StateGraph(State)
builder.add_node("detect_language", detect_language)
builder.add_node("korean_response", korean_response)
builder.add_node("english_response", english_response)

builder.add_edge(START, "detect_language")
builder.add_conditional_edges("detect_language", route_by_language)
builder.add_edge("korean_response", END)
builder.add_edge("english_response", END)

app = builder.compile()

## 3. 실행 따라가기

같은 그래프에 한국어·영어 질문을 하나씩 넣는다. **다음 ▶**을 누르며 노드가 실행되는 순서와 각 노드가 State에 쓴 값을 본다.

In [ ]:
show_graph_run(
    app,
    {
        "messages": [HumanMessage(content="안녕하세요! LangGraph에 대해 알려주세요.")],
        "language": "",
    },
)

In [ ]:
show_graph_run(
    app,
    {
        "messages": [HumanMessage(content="Hello! Tell me about LangGraph.")],
        "language": "",
    },
)

**볼 지점**
- 두 입력이 `detect_language` 뒤에서 갈라진다. 점선이 조건부 엣지다.
- `detect_language`는 `language`만 쓰고 `messages`는 건드리지 않는다. 응답 노드는 `messages`에 한 개를 더한다 — Reducer가 누적한다.

## 4. 슬라이드로 다시 정리

In [ ]:
show_slides("2-1", [7, 8, 10])

## 5. 리뷰 과제 — CodeMate에게 리뷰를 시키고, 직접 확인하기

**지시문 1 (리뷰만, 수정 금지)** — CodeMate 채팅에 붙여 넣는다.

> `detect_language` 함수는 `'ko' in result.content.lower()`로 언어를 판별한다. 이 방식이 틀린 결과를 낼 수 있는 모델 응답 예시 3개와, 그때 그래프가 어느 노드로 가는지 알려 줘. 코드는 아직 고치지 마.

CodeMate가 제시한 응답 예시를 아래 목록에 넣고 직접 확인한다. LLM을 부르지 않고 판별식만 검사한다.

In [ ]:
for fake_reply in ['ko', 'en', 'Korean', 'English', 'The text is English, not Korean.']:
    lang = 'ko' if 'ko' in fake_reply.lower() else 'en'
    print(f'{fake_reply!r:40} → {lang}')

**지시문 2 (수정)**

> 1.3에서 배운 `with_structured_output`을 써서 `detect_language`가 반드시 `'ko'` 또는 `'en'` 중 하나만 받도록 고쳐 줘. 노드 이름, 라우팅 함수, 엣지는 바꾸지 마.

고친 코드를 2번 셀에 반영하고 3번을 다시 실행한 뒤 아래로 리뷰한다.

In [ ]:
show_checklist([
    '노드 이름·라우팅 함수·엣지는 그대로다 (코드 지도의 구획 구성이 같다)',
    '모델 응답이 "ko" 또는 "en" 두 값으로 제한된다',
    '위 확인 셀의 오판 사례가 더 이상 생길 수 없는 이유를 설명할 수 있다',
    '3번 실행 따라가기에서 두 입력이 각각 맞는 응답 노드로 간다',
])